# Beginner Workshop: Single-Cell RNA-seq Analysis with Scanpy
## PBMC 3k Tutorial — Basics & Visualization

Welcome to this hands-on workshop on single-cell RNA sequencing (scRNA-seq) analysis using **Scanpy** and other tools from the [scverse](https://scverse.org/) ecosystem.

### Reference Tutorials
This workshop notebook is inspired by the following official Scanpy tutorials:
- [Preprocessing and clustering 3k PBMCs (legacy workflow)](https://scanpy.readthedocs.io/en/1.10.x/tutorials/basics/clustering-2017.html)
- [Preprocessing and clustering (modern workflow)](https://scanpy.readthedocs.io/en/1.10.x/tutorials/basics/clustering.html)
- [Integrating data using ingest](https://scanpy.readthedocs.io/en/1.10.x/tutorials/basics/integrating-data-using-ingest.html)
- [Core plotting functions](https://scanpy.readthedocs.io/en/1.10.x/tutorials/plotting/core.html)
- [Advanced plotting](https://scanpy.readthedocs.io/en/1.10.x/tutorials/plotting/advanced.html)

### Workshop Overview
| Section | Topic |
|---------|-------|
| 0 | Setup and imports |
| 1 | The AnnData data structure |
| 2 | Loading PBMC data |
| 3 | Exploring raw data |
| 4 | Quality control (QC) and filtering |
| 5 | Doublet detection |
| 6 | Normalization, HVGs, and scaling |
| 7 | Cell cycle scoring |
| 8 | Principal Component Analysis (PCA) |
| 9 | Clustering with Leiden |
| 10 | Embedding visualization (UMAP, t-SNE) |
| 11 | Visualization gallery |
| 12 | Marker gene detection |
| 13 | Cell type annotation |
| 14 | Data integration with ingest |
| 15 | Save results and workshop exercises |

### Dataset
We use the [PBMC 3k](https://support.10xgenomics.com/single-cell-gene-expression/datasets/1.1.0/pbmc3k) dataset — 2,700 peripheral blood mononuclear cells from a healthy donor sequenced on the 10x Chromium v1 platform. This is the standard Scanpy tutorial dataset.

> **Prerequisites:** basic Python knowledge. No prior scRNA-seq experience required.


## 0. Setup

### Running environment

For this workshop we will be using a **pre-built Apptainer image** that already has all required packages installed:

```
/oscar/data/shared/workshops/ccv_scrnaseq_2026.sif
```

See the main [README](../README.md) for step-by-step instructions on launching the notebook through the Oscar OpenOnDemand portal.

> **Running outside the Apptainer image?**  
> If you need to run this notebook on Oscar without the container, or on your own machine, you will need to create a virtual environment and install all packages manually.  
> Follow the [Oscar Python virtual environment guide](https://docs.ccv.brown.edu/oscar/software/python-installs) for instructions on setting up a `venv` on Oscar, then install the packages:
> ```bash
> pip install -r requirements.txt
> ```
> *(A `requirements.txt` listing all workshop packages is provided at the root of this repository.)*

### Key packages used in this notebook

- **scanpy** — single-cell analysis in Python
- **anndata** — annotated data matrix (the core data structure)
- **matplotlib / seaborn** — plotting
- **scrublet** — doublet detection
- **pyhere** — project-root-relative path resolution
- **harmonypy** — Harmony batch integration
- **scvi-tools** — scVI deep generative model integration
- **gseapy** — Gene Set Enrichment Analysis & Over-Representation Analysis


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata as ad
from pyhere import here

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Project-relative paths ───────────────────────────────────────────
# here() walks up from cwd until it finds .here / .git / setup.py
# and returns an absolute pathlib.Path — works regardless of where
# the notebook is run from after cloning the repository.
DATA_DIR    = here('data')     # data/   — downloaded datasets & outputs
FIGURES_DIR = here('figures')  # figures/ — saved plots
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

# Scanpy global settings
sc.settings.verbosity   = 3         # 0=errors only … 3=hints
sc.settings.n_jobs      = 1         # sets number of CPU cores, increase for large datasets
sc.settings.datasetdir  = DATA_DIR  # downloaded datasets land here
sc.settings.figdir      = str(FIGURES_DIR)  # sc.pl. save= writes here
sc.set_figure_params(dpi=100, facecolor='white', figsize=(5, 4))

print('scanpy version:', sc.__version__)
print('anndata version:', ad.__version__)
print('project root:', here())
print('data dir:    ', DATA_DIR)
print('figures dir: ', FIGURES_DIR)


## 1. The AnnData Data Structure

Almost everything in Scanpy revolves around an **AnnData** object. Think of it as a smart container that holds the expression matrix together with all associated metadata.

```
              genes (var)
          ┌────────────────────────┐
   cells  │                        │
   (obs)  │    X  (expression      │
          │       matrix)          │
          └────────────────────────┘
```

| Slot | What it holds |
|------|---------------|
| `adata.X` | count / expression matrix (cells × genes) |
| `adata.obs` | cell metadata (e.g., QC metrics, cluster labels) |
| `adata.var` | gene metadata (e.g., HVG flags, mean expression) |
| `adata.obsm` | cell-level embeddings (e.g., PCA, UMAP) |
| `adata.obsp` | pairwise cell relationships (e.g., kNN graph) |
| `adata.uns` | unstructured metadata (colors, neighbor params, etc.) |
| `adata.layers` | additional expression matrices (raw counts, batch-corrected) |
| `adata.raw` | frozen snapshot before HVG-subsetting |

> **Workshop question:** Why is it useful to keep all this information in one container?


## 2. Loading PBMC Data

### Option A — Built-in dataset (recommended for this workshop)
Scanpy ships with the PBMC 3k dataset via `sc.datasets.pbmc3k()`.  
Because we set `sc.settings.datasetdir = DATA_DIR` above, the downloaded
files are stored inside the project's `data/` folder automatically.

### Option B — Load from 10x CellRanger output
In a real project you would load your own data from a CellRanger output
directory. Using `here()` keeps the path portable:
```python
# Download and unpack the raw data:
# mkdir -p data && cd data
# curl https://cf.10xgenomics.com/samples/cell/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz \
#      -o pbmc3k_filtered_gene_bc_matrices.tar.gz
# tar -xzf pbmc3k_filtered_gene_bc_matrices.tar.gz

adata = sc.read_10x_mtx(
    here('data', 'filtered_gene_bc_matrices', 'hg19'),
    var_names='gene_symbols',
    cache=True,
)
```


In [ ]:
# Load PBMC3k using the built-in helper (downloads automatically)
adata = sc.datasets.pbmc3k()
adata.var_names_make_unique()   # ensure all gene names are unique

print('Shape:', adata.shape)    # (n_cells, n_genes)
print(adata)


In [ ]:
# Inspect the cell and gene metadata tables
print('obs (cell metadata):'); print(adata.obs.head())
print()
print('var (gene metadata):'); print(adata.var.head())


In [ ]:
# Preserve raw counts in a separate layer for later use
adata.layers['counts'] = adata.X.copy()

# Output file path — resolved relative to the project root via here()
results_file = here('data', 'pbmc3k_workshop.h5ad')
print('Results will be saved to:', results_file)


## 3. Exploring Raw Data

Before any filtering it is useful to visualise which genes dominate the total counts. The plot below shows the top 20 genes ranked by mean fraction of counts per cell.

> **Tip:** ribosomal proteins and mitochondrial genes often appear at the top.


In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20)


## 4. Quality Control and Filtering

Before analysis we remove **low-quality cells** and uninformative genes.

### Key QC metrics
| Metric | What it measures | Problem indicated |
|--------|-----------------|-------------------|
| `n_genes_by_counts` | Number of detected genes | Low → empty droplet; very high → doublet |
| `total_counts` | Total UMIs (library size) | Closely correlated with gene count |
| `pct_counts_mt` | % of counts from mt genes | High → damaged/dying cell |
| `pct_counts_ribo` | % of counts from ribo genes | Very high may indicate stress |

Mitochondrial gene names start with **`MT-`** (human) or **`mt-`** (mouse).


In [ ]:
# Flag mitochondrial and ribosomal genes
adata.var['mt']   = adata.var_names.str.startswith('MT-')
adata.var['ribo'] = adata.var_names.str.startswith(('RPS', 'RPL'))

# Compute per-cell QC metrics
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=['mt', 'ribo'],
    percent_top=None,
    log1p=False,
    inplace=True,
)

print('QC columns added to adata.obs:')
print([c for c in adata.obs.columns])


In [ ]:
# QC summary tables (5% quantiles + dynamic mito threshold)
quantiles = np.linspace(0, 1, 21)
qc_metrics = ['total_counts', 'n_genes_by_counts', 'pct_counts_mt', 'pct_counts_ribo']

qc_quantiles = adata.obs[qc_metrics].quantile(quantiles)
qc_quantiles.index = [f'{int(q * 100)}%' for q in qc_quantiles.index]

qc_quantiles_display = qc_quantiles.copy()
qc_quantiles_display[['total_counts', 'n_genes_by_counts']] = (
    qc_quantiles_display[['total_counts', 'n_genes_by_counts']].round(0).astype(int)
)
qc_quantiles_display[['pct_counts_mt', 'pct_counts_ribo']] = (
    qc_quantiles_display[['pct_counts_mt', 'pct_counts_ribo']].round(3)
)

print('5% quantile table for initial QC metrics')
display(
    qc_quantiles_display.rename(
        columns={
            'total_counts': 'n_counts (total_counts)',
            'n_genes_by_counts': 'n_features (n_genes_by_counts)',
            'pct_counts_mt': 'percent mito (pct_counts_mt)',
            'pct_counts_ribo': 'percent ribosomal (pct_counts_ribo)',
        }
    )
)

pct_counts_mt_mean = adata.obs['pct_counts_mt'].mean()
pct_counts_mt_std = adata.obs['pct_counts_mt'].std()
pct_counts_mt_mean_plus_2sd = pct_counts_mt_mean + (2 * pct_counts_mt_std)

print('Suggested dynamic mito cutoff (for filtering): mean + 2 × SD')
display(
    pd.DataFrame(
        {
            'metric': [
                'pct_counts_mt mean',
                'pct_counts_mt standard deviation',
                'pct_counts_mt mean + 2 × SD (suggested filter cutoff)',
            ],
            'value': [pct_counts_mt_mean, pct_counts_mt_std, pct_counts_mt_mean_plus_2sd],
        }
    ).set_index('metric').round(3)
)

# Example dynamic filter:
# adata = adata[adata.obs.pct_counts_mt < pct_counts_mt_mean_plus_2sd, :].copy()


In [ ]:
# Violin plots — overview of three key QC metrics
sc.pl.violin(
    adata,
    ['n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo'],
    jitter=0.4,
    multi_panel=True,
)


In [ ]:
# Scatter plots — visually identify outlier cells
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt', ax=axes[0], show=False)
axes[0].axhline(5, color='red', linestyle='--', label='5% MT threshold')
axes[0].axvline(7500, color='green', linestyle='--', label='7500 total_counts cutoff')
axes[0].legend()

sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', ax=axes[1], show=False)
axes[1].axhline(2500, color='red', linestyle='--', label='2500 gene threshold')
axes[1].axvline(7500, color='green', linestyle='--', label='7500 total_counts cutoff')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Scatter plots with data-driven (dynamic) cutoffs
# Thresholds computed by the preceding QC summary cell (qc_quantiles, pct_counts_mt_mean_plus_2sd)
_tc_q05  = qc_quantiles.loc['5%',  'total_counts']
_tc_q95  = qc_quantiles.loc['95%', 'total_counts']
_ng_q05  = qc_quantiles.loc['5%',  'n_genes_by_counts']
_ng_q95  = qc_quantiles.loc['95%', 'n_genes_by_counts']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Scatter plots — dynamic QC thresholds', fontsize=12)

# Plot 1: total_counts vs pct_counts_mt
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt', ax=axes[0], show=False)
axes[0].axhline(pct_counts_mt_mean_plus_2sd, color='red',   linestyle='--',
                label=f'mito mean+2SD ({pct_counts_mt_mean_plus_2sd:.2f}%)')
axes[0].axvline(_tc_q05, color='blue',  linestyle=':',
                label=f'total_counts 5% ({_tc_q05:.0f})')
axes[0].axvline(_tc_q95, color='blue',  linestyle='-.',
                label=f'total_counts 95% ({_tc_q95:.0f})')
axes[0].set_title('total_counts vs pct_counts_mt')
axes[0].legend(fontsize=7)

# Plot 2: total_counts vs n_genes_by_counts
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', ax=axes[1], show=False)
axes[1].axvline(_tc_q05, color='blue',  linestyle=':',
                label=f'total_counts 5% ({_tc_q05:.0f})')
axes[1].axvline(_tc_q95, color='blue',  linestyle='-.',
                label=f'total_counts 95% ({_tc_q95:.0f})')
axes[1].axhline(_ng_q05, color='orange', linestyle=':',
                label=f'n_genes 5% ({_ng_q05:.0f})')
axes[1].axhline(_ng_q95, color='orange', linestyle='-.',
                label=f'n_genes 95% ({_ng_q95:.0f})')
axes[1].set_title('total_counts vs n_genes_by_counts')
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()


### 💭 Thought Exercise: Choosing QC Thresholds

You have just seen two sets of scatter plots:

| | Fixed-threshold plot | Dynamic-threshold plot |
|---|---|---|
| **MT%** | 5% hard cutoff | mean + 2 × SD |
| **total_counts (upper)** | 7,500 | 95th percentile |
| **total_counts (lower)** | *(none)* | 5th percentile |
| **n_genes** | 2,500 hard cutoff | 5th / 95th percentile |

Take a moment to compare the two before proceeding to filtering:

1. **Consistency check** — Do the fixed thresholds (5%, 7,500, 2,500) sit inside the data-driven bands, or do they fall outside the 5th–95th percentile range? What does that tell you?

2. **Strictness** — For *this* dataset, which approach removes more cells? Can you tell from the plots without running the filters? Which would you choose for a new dataset you had never seen before?

3. **Mitochondrial reads** — The mito threshold differs between the two plots. How would you decide between a biology-based rule (e.g., '5% is standard for PBMCs') and a statistics-based rule (mean + 2 × SD)? What are the risks of each?

4. **Edge cases** — Look at cells near `total_counts = 7,500` in the first plot. Are they also flagged by the data-driven thresholds? What would happen to those cells under each filtering strategy?

5. **Your dataset** — Suppose you received a brand-new dataset from a different tissue type (e.g., brain or liver). Which approach would you start with, and what additional information would help you refine the thresholds?

> **Key takeaway:** Fixed thresholds are easy to communicate and reproduce, but data-driven thresholds adapt to the dataset. In practice, bioinformaticians often start with data-driven exploration and then settle on fixed thresholds that they can defend biologically.


### 4.1 Doublet Detection with Scrublet

**Doublets** are droplets that accidentally captured two cells. They appear as outliers in
gene-expression space and can produce spurious clusters, inflated marker lists, and incorrect
cell-type assignments.

[Scrublet](https://github.com/swolock/scrublet) detects doublets by simulating synthetic ones:
it randomly combines pairs of real cells and scores each observed cell by how closely it
resembles those simulated doublets.

#### ⚠️ Why run doublet detection *before* QC filtering?

Scrublet's simulation step samples pairs from **the full, unfiltered count matrix**. If you
filter first, you remove some real cells — including cells that genuinely resemble doublets —
and the simulated doublet distribution becomes narrower and less representative. This leads to:

* **Underestimation** of doublet scores for remaining cells (the reference distribution shifts).
* **Missed doublets** that would have been caught by a wider simulated distribution.
* Results that are harder to reproduce across datasets.

The recommended order is therefore:

1. Compute QC metrics on all cells (already done in § 4).
2. Calculate filtering thresholds from the full distribution (already done in § 4).
3. **Run doublet detection on unfiltered data** (this cell).
4. Apply QC filters *and* remove predicted doublets together (§ 4.2).

> **Note:** Scrublet expects **raw counts**. We use the `counts` layer saved earlier.


In [ ]:
try:
    import scrublet as scr

    # Run on the full, unfiltered count matrix so the simulated doublet
    # distribution is as representative as possible.
    scrub = scr.Scrublet(adata.layers['counts'])
    doublet_scores, predicted_doublets = scrub.scrub_doublets()

    # Store results in obs — we will use these when filtering in § 4.2
    adata.obs['doublet_score'] = doublet_scores
    adata.obs['predicted_doublet'] = predicted_doublets

    scrub.plot_histogram()
    plt.show()

    n_d = predicted_doublets.sum()
    print(f'Predicted doublets: {n_d} / {adata.n_obs} ({100*n_d/adata.n_obs:.1f}%)')
    print('Doublets will be removed together with low-quality cells in § 4.2.')

except ImportError:
    print('scrublet not installed — skipping. Install with: pip install scrublet')
    adata.obs['doublet_score'] = float('nan')
    adata.obs['predicted_doublet'] = False


### 4.2 Apply Filters

We now apply all cell-level filters in a single pass, using the **data-driven thresholds**
computed from the full pre-filter distribution in § 4, plus the doublet labels from § 4.1:

| Filter | Threshold | Rationale |
|---|---|---|
| min genes per cell | 200 | removes empty droplets |
| min genes per cell | 5th percentile of `n_genes_by_counts` | removes low-complexity cells |
| max genes per cell | 95th percentile of `n_genes_by_counts` | removes likely multiplets |
| total counts (lower) | 5th percentile of `total_counts` | removes very low-depth cells |
| total counts (upper) | 95th percentile of `total_counts` | removes potential multiplets |
| mitochondrial % | mean + 2 × SD of `pct_counts_mt` | removes damaged / dying cells |
| predicted doublet | `False` | removes Scrublet-flagged doublets |
| min cells per gene | 3 | removes uninformative genes |

> **Note:** All numeric thresholds were calculated from the *unfiltered* dataset so they reflect
> the true data distribution and are not biased by prior filtering.


In [ ]:
print(f'Before filtering: {adata.n_obs} cells, {adata.n_vars} genes')

# 1. Remove cells with fewer than 200 detected genes (empty droplets)
sc.pp.filter_cells(adata, min_genes=200)

# 2. Remove genes detected in fewer than 3 cells (uninformative genes)
sc.pp.filter_genes(adata, min_cells=3)

# 3. Data-driven (dynamic) cell-level filters
#    Variables computed in § 4 on the full pre-filter distribution:
#      _tc_q05 / _tc_q95            : 5th / 95th percentile of total_counts
#      _ng_q05 / _ng_q95            : 5th / 95th percentile of n_genes_by_counts
#      pct_counts_mt_mean_plus_2sd  : mean + 2 × SD of pct_counts_mt
#    Plus doublet labels from § 4.1.
adata = adata[
    (adata.obs.total_counts      >= _tc_q05) &
    (adata.obs.total_counts      <= _tc_q95) &
    (adata.obs.n_genes_by_counts >= _ng_q05) &
    (adata.obs.n_genes_by_counts <= _ng_q95) &
    (adata.obs.pct_counts_mt     <  pct_counts_mt_mean_plus_2sd) &
    (~adata.obs.predicted_doublet),
    :
].copy()

print(f'After filtering:  {adata.n_obs} cells, {adata.n_vars} genes')
print(f'  total_counts   : [{_tc_q05:.0f}, {_tc_q95:.0f}]')
print(f'  n_genes        : [{_ng_q05:.0f}, {_ng_q95:.0f}]')
print(f'  pct_counts_mt  : < {pct_counts_mt_mean_plus_2sd:.3f}%')
print('  predicted_doublet: removed')


## 5. Normalization, HVG Selection, and Scaling

### 5.1 Normalization

Cells are sequenced to different depths (some deeper than others). We equalise them by:
1. **Total-count normalization** — scale each cell to 10,000 UMIs
2. **log1p transformation** — `log(x + 1)`, compresses dynamic range and stabilises variance


In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Save a snapshot of normalised, log-transformed data before HVG subsetting.
# adata.raw retains all genes for downstream visualisation.
adata.raw = adata

print('Normalization complete. adata.raw holds the full gene expression matrix.')


### 5.2 Highly Variable Genes (HVGs)

Most genes vary little across cells — they add noise rather than signal. We select the **highly variable genes** (those that vary more than expected by chance) to focus downstream analyses.

Scanpy's default method (Seurat v1): genes are binned by mean expression; within each bin the normalised dispersion (variance/mean) is computed; genes with dispersion above a threshold are selected.


In [ ]:
sc.pp.highly_variable_genes(
    adata,
    min_mean=0.0125,
    max_mean=3,
    min_disp=0.5,
)

print(f'HVGs: {adata.var.highly_variable.sum()} / {adata.n_vars}')
sc.pl.highly_variable_genes(adata)


### 5.3 Regress Out Confounders and Scale

We **regress out** total UMI counts and mitochondrial fraction so they do not dominate PCA. Then we **scale** each gene to unit variance (z-score) so all genes contribute equally.

> **Note:** We first subset to HVGs (keeping `.raw` for full-gene visualisation), then regress and scale.


In [ ]:
# Subset to HVGs only; adata.raw still holds all genes
adata = adata[:, adata.var.highly_variable].copy()

# Regress out technical confounders
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])

# Scale to unit variance; clip extreme outliers at 10 SDs
sc.pp.scale(adata, max_value=10)

print('Preprocessing done. Shape after HVG subset:', adata.shape)


## 6. Cell Cycle Scoring

Cell-cycle-driven variation can confound clustering. We score each cell for its **S-phase** and **G2/M-phase** activity using the gene lists from Tirosh et al. 2016.

> **Note:** For this PBMC dataset, cell-cycle effects are modest. They are more prominent in actively cycling datasets (tumour, stem cells, etc.).


In [ ]:
# Tirosh et al. 2016 cell cycle gene lists
s_genes = [
    'MCM5', 'PCNA', 'TYMS', 'FEN1', 'MCM2', 'MCM4', 'RRM1', 'UNG',
    'GINS2', 'MCM6', 'CDCA7', 'DTL', 'PRIM1', 'UHRF1', 'MLF1IP',
    'HELLS', 'RFC2', 'RPA2', 'NASP', 'RAD51AP1', 'GMNN', 'WDR76',
    'SLBP', 'CCNE2', 'UBR7', 'POLD3', 'MSH2', 'ATAD2', 'RAD51',
    'RRM2', 'CDC45', 'CDC6', 'EXO1', 'TIPIN', 'DSCC1', 'BLM',
    'CASP8AP2', 'USP1', 'CLSPN', 'POLA1', 'CHAF1B', 'BRIP1', 'E2F8',
]

g2m_genes = [
    'HMGB2', 'CDK1', 'NUSAP1', 'UBE2C', 'BIRC5', 'TPX2', 'TOP2A',
    'NDC80', 'CKS2', 'NUF2', 'CKS1B', 'MKI67', 'TMPO', 'CENPF',
    'TACC3', 'FAM64A', 'SMC4', 'CCNB2', 'CKAP2L', 'CKAP2', 'AURKB',
    'BUB1', 'KIF11', 'ANP32E', 'TUBB4B', 'GTSE1', 'KIF20B', 'HJURP',
    'CDCA3', 'HN1', 'CDC20', 'TTK', 'CDC25C', 'KIF2C', 'RANGAP1',
    'NCAPD2', 'DLGAP5', 'CDCA2', 'CDCA8', 'ECT2', 'KIF23', 'HMMR',
    'AURKA', 'PSRC1', 'ANLN', 'LBR', 'CKAP5', 'CENPE', 'CTCF',
    'NEK2', 'G2E3', 'GAS2L3', 'CBX5', 'CENPA',
]

# Filter to genes present in our HVG-subset data
s_use   = [g for g in s_genes   if g in adata.var_names]
g2m_use = [g for g in g2m_genes if g in adata.var_names]

if s_use and g2m_use:
    sc.tl.score_genes_cell_cycle(adata, s_genes=s_use, g2m_genes=g2m_use)
    print('Cell cycle phase distribution:')
    print(adata.obs['phase'].value_counts())
else:
    print('Not enough cell cycle genes in HVG subset — skipping.')


## 7. Principal Component Analysis (PCA)

PCA projects the ~2 000-gene HVG space into a smaller set of **principal components (PCs)** that capture the most variation.

### Steps
1. Compute PCs
2. Inspect the **elbow plot** to choose how many PCs to use
3. Visualise cells in PC space

> **Tip:** The elbow plot shows variance explained per PC. The 'elbow' — where the curve flattens — is a common heuristic for choosing the number of PCs.


In [ ]:
sc.tl.pca(adata, n_comps=50)  # svd_solver='auto' is the default and works well for typical HVG subset sizes

# Elbow plot — variance explained per PC
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50)


In [ ]:
# Scatter plot in PC space, coloured by QC metrics
sc.pl.pca(
    adata,
    color=['total_counts', 'pct_counts_mt'],
    ncols=2,
)


In [ ]:
# Colour by cell cycle phase if scored
if 'phase' in adata.obs.columns:
    sc.pl.pca(adata, color='phase')


> **Workshop question:** How many PCs capture most of the meaningful biological variation in the elbow plot?


## 8. Clustering with the Leiden Algorithm

Graph-based clustering in two steps:
1. **Build a k-nearest-neighbour (kNN) graph** in PC space
2. **Detect communities** using the Leiden algorithm

The `resolution` parameter controls granularity: higher → more clusters.

> **Workshop note:** We compute UMAP here too so we can visualise clusters immediately below.


In [ ]:
# Build kNN graph (40 PCs, 10 neighbours)
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

# Compute UMAP
sc.tl.umap(adata)

# Leiden clustering at three resolutions
for res in [0.3, 0.5, 1.0]:
    key = f'leiden_res{res}'
    sc.tl.leiden(adata, resolution=res, key_added=key)
    print(f'Leiden res={res}: {adata.obs[key].nunique()} clusters')


## 9. Embedding Visualisations (UMAP & t-SNE)

### 9.1 UMAP

UMAP (Uniform Manifold Approximation and Projection) projects cells into 2D while preserving local neighbourhood structure. It is the most widely used embedding in single-cell analysis.

> **Important:** UMAP distances are *not* globally meaningful — do not over-interpret the positions of distant clusters.


In [ ]:
# Clustering at three resolutions side by side
sc.pl.umap(
    adata,
    color=['leiden_res0.3', 'leiden_res0.5', 'leiden_res1.0'],
    ncols=3,
    legend_loc='on data',
    title=['Leiden (res=0.3)', 'Leiden (res=0.5)', 'Leiden (res=1.0)'],
)


In [ ]:
# QC metrics on UMAP — check if any cluster is low quality
sc.pl.umap(
    adata,
    color=['total_counts', 'n_genes_by_counts', 'pct_counts_mt'],
    ncols=3,
    title=['Total UMI counts', 'Genes per cell', '% MT reads'],
)


In [ ]:
# Canonical PBMC marker genes on UMAP
# (using .raw so we show all genes, not just HVGs)
marker_genes_preview = {
    'CD3D':  'T cells',
    'CD19':  'B cells',
    'CD14':  'Monocytes',
    'GNLY':  'NK cells',
    'FCER1A':'Dendritic cells',
    'PPBP':  'Megakaryocytes',
}
genes_present = [g for g in marker_genes_preview if g in adata.raw.var_names]
sc.pl.umap(
    adata,
    color=genes_present,
    use_raw=True,
    ncols=3,
    title=[f'{g} ({marker_genes_preview[g]})' for g in genes_present],
)


### 9.2 t-SNE

t-SNE is an alternative 2D embedding that tends to produce tighter clusters but is slower and less scalable than UMAP.


In [ ]:
sc.tl.tsne(adata, use_rep='X_pca', learning_rate='auto')
sc.pl.tsne(adata, color='leiden_res0.5', legend_loc='on data')


## 10. Visualization Gallery

Scanpy provides a rich set of plotting functions for exploring gene expression patterns. We showcase the main ones here.

We use clusters at **res=0.5** and a panel of canonical PBMC marker genes.


In [ ]:
# Canonical PBMC marker gene panels (grouped by cell type)
pbmc_markers = {
    'CD4 T': ['IL7R', 'CCR7', 'S100A4'],
    'CD8 T': ['CD8A', 'CD8B'],
    'B':     ['CD19', 'MS4A1', 'CD79A'],
    'NK':    ['GNLY', 'NKG7', 'GZMB'],
    'Mono':  ['CD14', 'LYZ', 'CST3'],
    'DC':    ['FCER1A', 'CST3'],
    'Plt':   ['PPBP'],
}

# Flatten to unique genes that are actually in the dataset
all_markers = list(dict.fromkeys(
    g for genes in pbmc_markers.values() for g in genes
    if g in adata.raw.var_names
))
print('Marker genes available:', all_markers)


### 10.1 Dot Plot

Shows the **fraction of cells** expressing a gene (dot size) and the **mean expression level** (colour) per group.


In [ ]:
sc.pl.dotplot(
    adata,
    var_names=pbmc_markers,
    groupby='leiden_res0.5',
    dendrogram=True,
    use_raw=True,
    standard_scale='var',
    title='Marker gene expression per cluster',
)


### 10.2 Violin Plot

Shows the **distribution of expression** per cluster for individual genes.


In [ ]:
sc.pl.violin(
    adata,
    keys=all_markers[:6],
    groupby='leiden_res0.5',
    rotation=45,
    use_raw=True,
)


### 10.3 Heatmap

Shows mean expression per cluster as a colour grid. Useful for a quick overview of many markers at once.


In [ ]:
sc.pl.heatmap(
    adata,
    var_names=pbmc_markers,
    groupby='leiden_res0.5',
    use_raw=True,
    standard_scale='var',
    dendrogram=True,
    cmap='viridis',
)


### 10.4 Matrix Plot

Similar to a heatmap but averages expression within groups and uses a diverging colour scale.


In [ ]:
sc.pl.matrixplot(
    adata,
    var_names=pbmc_markers,
    groupby='leiden_res0.5',
    use_raw=True,
    standard_scale='var',
    dendrogram=True,
    cmap='RdBu_r',
)


### 10.5 Stacked Violin Plot

Combines violin plots for multiple genes into a compact stacked layout.


In [ ]:
sc.pl.stacked_violin(
    adata,
    var_names=pbmc_markers,
    groupby='leiden_res0.5',
    use_raw=True,
    dendrogram=True,
)


### 10.6 Tracks Plot

Shows expression as horizontal bars for each cell, sorted by cluster. Excellent for seeing within-cluster heterogeneity.


In [ ]:
sc.pl.tracksplot(
    adata,
    var_names=all_markers[:8],
    groupby='leiden_res0.5',
    use_raw=True,
)


## 11. Marker Gene Detection

To understand what each cluster represents we perform **differential expression** — finding genes that are highly expressed in one cluster compared to all others (one-vs-rest).

### Available tests
| Method | Notes |
|--------|-------|
| `wilcoxon` | Non-parametric; robust; **recommended** |
| `t-test` | Fast; assumes normality |
| `logreg` | Logistic regression; useful with many clusters |


In [ ]:
# Rank genes using Wilcoxon test; use .raw (all genes)
sc.tl.rank_genes_groups(
    adata,
    groupby='leiden_res0.5',
    method='wilcoxon',
    use_raw=True,
    n_genes=25,
)

# Overview panel: top 15 markers per cluster
sc.pl.rank_genes_groups(adata, n_genes=15, sharey=False)


In [ ]:
# Table of top markers
marker_df = sc.get.rank_genes_groups_df(adata, group=None)
print(marker_df.head(20))


In [ ]:
# Dot plot: top 3 marker genes per cluster
sc.pl.rank_genes_groups_dotplot(
    adata,
    n_genes=3,
    use_raw=True,
    standard_scale='var',
)


In [ ]:
# Violin: expression of top markers for clusters 0, 1, 2
sc.pl.rank_genes_groups_violin(
    adata,
    groups=['0', '1', '2'],
    n_genes=5,
    use_raw=True,
)


## 12. Cell Type Annotation

We annotate each cluster by comparing its marker genes with published PBMC signatures.

### Canonical PBMC marker genes
| Cell type | Key markers |
|-----------|------------|
| CD4+ T cells | IL7R, CCR7, S100A4 |
| CD8+ T cells | CD8A, CD8B |
| B cells | CD19, MS4A1, CD79A |
| NK cells | GNLY, NKG7, GZMB |
| CD14+ Monocytes | CD14, LYZ, CST3 |
| FCGR3A+ Monocytes | FCGR3A, MS4A7 |
| Dendritic cells | FCER1A, CST3 |
| Megakaryocytes | PPBP |

> **Workshop exercise:** Compare your cluster markers (Section 12) with the table above to determine each cluster's identity.


In [ ]:
# Example annotation mapping
# NOTE: cluster numbers vary between runs — adjust to match your own results!
cluster_to_celltype = {
    '0': 'CD4 T',
    '1': 'CD14+ Mono',
    '2': 'CD4 T',
    '3': 'NK / CD8 T',
    '4': 'B',
    '5': 'CD8 T',
    '6': 'FCGR3A+ Mono',
    '7': 'NK',
    '8': 'DC',
    '9': 'Platelet',
}

adata.obs['cell_type'] = (
    adata.obs['leiden_res0.5']
         .map(cluster_to_celltype)
         .fillna('Unknown')
         .astype('category')
)

sc.pl.umap(
    adata,
    color='cell_type',
    legend_loc='on data',
    title='Annotated PBMC cell types',
    frameon=False,
)


In [ ]:
# Dot plot with annotated cell types
sc.pl.dotplot(
    adata,
    var_names=pbmc_markers,
    groupby='cell_type',
    use_raw=True,
    standard_scale='var',
    dendrogram=True,
)


In [ ]:
# Bar chart of cell type proportions
cell_counts = adata.obs['cell_type'].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
cell_counts.plot.bar(ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Cell type')
ax.set_ylabel('Number of cells')
ax.set_title('Cell type composition — PBMC 3k')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 13. Data Integration with `sc.tl.ingest`

*Reference: [Integrating data using ingest and BBKNN](https://scanpy.readthedocs.io/en/1.10.x/tutorials/basics/integrating-data-using-ingest.html)*

When you have a well-annotated **reference dataset** and a new **query dataset**, you can transfer labels from the reference to the query using `sc.tl.ingest`.

### How it works
1. Fit a PCA + kNN model on the reference
2. Project query cells into the reference's PCA space
3. Assign each query cell the label of its nearest neighbour in the reference

**Advantages over batch correction approaches (Harmony, BBKNN, etc.)**:
- Transparent and fast
- Solves the label-transfer problem directly
- Maintains the reference embedding structure

> **Note:** This asymmetric approach (*ingesting* annotations from reference → query) is different from jointly integrating datasets.


In [ ]:
# Load the pre-processed PBMC3k as reference
# (Scanpy's built-in processed version already has cell type labels)
adata_ref = sc.datasets.pbmc3k_processed()

# Load the PBMC 68k reduced dataset as query
adata_query = sc.datasets.pbmc68k_reduced()

print('Reference:', adata_ref.shape, '| Labels:', adata_ref.obs['louvain'].unique().tolist())
print('Query:    ', adata_query.shape)


In [ ]:
# ingest requires datasets to share the same variable names
var_names = adata_ref.var_names.intersection(adata_query.var_names)
adata_ref   = adata_ref[:, var_names].copy()
adata_query = adata_query[:, var_names].copy()

print('Shared genes:', len(var_names))


In [ ]:
# Train the PCA/kNN model on the reference
sc.pp.pca(adata_ref)
sc.pp.neighbors(adata_ref)
sc.tl.umap(adata_ref)

# Check the reference UMAP
sc.pl.umap(adata_ref, color='louvain', title='Reference PBMC3k (annotated)')


In [ ]:
# Ingest: project query cells into reference space and transfer 'louvain' labels
sc.tl.ingest(adata_query, adata_ref, obs='louvain')

# Visualise query cells projected onto the reference UMAP
sc.pl.umap(
    adata_query,
    color=['louvain'],
    title='Query PBMC68k — transferred labels',
)


In [ ]:
# Concatenate reference and query for a combined view
adata_combined = adata_ref.concatenate(
    adata_query,
    batch_categories=['3k', '68k'],
)

sc.pl.umap(
    adata_combined,
    color=['louvain', 'batch'],
    title=['Cell type (transferred)', 'Dataset batch'],
    ncols=2,
)


## 14. Integration Methods Comparison: Harmony vs scVI

When analysing cells from multiple samples or experiments, **batch effects** — technical differences unrelated to biology — can dominate the signal.  Two leading integration strategies are:

| Method | Approach | Key property |
|--------|----------|--------------|
| **Harmony** | Iteratively adjusts PCA coordinates to align batches | Fast; works in PCA embedding space |
| **scVI** | Deep generative model trained directly on raw counts | More expressive; handles complex batch structures |

In this section we:
1. Reload PBMC 3k with raw counts and assign two **artificial batches** (random 50/50 split) with a simulated expression shift in batch B
2. Run standard HVG → PCA preprocessing
3. Apply **Harmony** via `harmonypy`
4. Apply **scVI** via `scvi-tools`
5. Compare UMAPs (uncorrected vs Harmony vs scVI) coloured by batch and by cluster

> **Install:** `pip install harmonypy scvi-tools`


In [ ]:
import scipy.sparse as sp

try:
    import harmonypy as hm
    import scvi
    _integration_ok = True
except ImportError:
    print("Missing packages — run: pip install harmonypy scvi-tools")
    _integration_ok = False

if _integration_ok:
    # ── 1. Load raw PBMC3k and apply the same QC filters as earlier ───────────
    adata_int = sc.datasets.pbmc3k()
    adata_int.var_names_make_unique()
    sc.pp.filter_cells(adata_int, min_genes=200)
    sc.pp.filter_genes(adata_int, min_cells=3)
    adata_int.var['mt'] = adata_int.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata_int, qc_vars=['mt'], inplace=True)
    adata_int = adata_int[
        (adata_int.obs.n_genes_by_counts < 2500) &
        (adata_int.obs.pct_counts_mt < 5)
    ].copy()

    # ── 2. Assign random artificial batches ───────────────────────────────────
    _rng = np.random.default_rng(RANDOM_SEED)
    adata_int.obs['batch'] = pd.Categorical(
        _rng.choice(['batch_A', 'batch_B'], size=adata_int.n_obs)
    )

    # Store raw integer counts — scVI requires these in a named layer
    adata_int.layers['counts'] = adata_int.X.copy()

    # ── 3. Simulate batch effect: add Poisson noise to batch_B raw counts ─────
    # This mimics the kind of systematic count inflation seen in some library-prep batches.
    _mask_b = (adata_int.obs['batch'] == 'batch_B').values
    _X = adata_int.layers['counts'].toarray().astype(np.float32)
    _X[_mask_b, :] += _rng.poisson(3, size=(_mask_b.sum(), adata_int.n_vars)).astype(np.float32)
    adata_int.layers['counts'] = sp.csr_matrix(_X)
    adata_int.X = adata_int.layers['counts'].copy()

    # ── 4. Normalise + log-transform (for Harmony / PCA) ─────────────────────
    sc.pp.normalize_total(adata_int, target_sum=1e4)
    sc.pp.log1p(adata_int)

    # ── 5. Batch-aware HVG selection + scale + PCA ────────────────────────────
    sc.pp.highly_variable_genes(
        adata_int, n_top_genes=2000,
        batch_key='batch', flavor='seurat_v3', layer='counts'
    )
    adata_int = adata_int[:, adata_int.var['highly_variable']].copy()
    sc.pp.scale(adata_int, max_value=10)
    sc.tl.pca(adata_int, n_comps=30)

    # Uncorrected kNN graph + UMAP (baseline: batch effect visible)
    sc.pp.neighbors(adata_int, n_pcs=30)
    sc.tl.umap(adata_int)
    adata_int.obsm['X_umap_uncorrected'] = adata_int.obsm['X_umap'].copy()
    sc.tl.leiden(adata_int, resolution=0.5, key_added='leiden_uncorrected')

    print(f"Dataset: {adata_int.n_obs} cells x {adata_int.n_vars} HVGs")
    print(f"  batch_A: {(adata_int.obs['batch']=='batch_A').sum()} cells")
    print(f"  batch_B: {(adata_int.obs['batch']=='batch_B').sum()} cells")


In [ ]:
if _integration_ok:
    # ── Harmony integration ───────────────────────────────────────────────────
    # Harmony adjusts the PCA embedding so that cells from different batches
    # that are biologically similar land close together.

    print("Running Harmony integration...")
    _ho = hm.run_harmony(
        adata_int.obsm['X_pca'],
        adata_int.obs,
        'batch',
        random_state=RANDOM_SEED,
    )
    # Store the corrected PCA coordinates
    Z = _ho.Z_corr
    if not isinstance(Z, np.ndarray):
        Z = np.array(Z)
    if Z.shape[0] != adata_int.n_obs:
        Z = Z.T
    adata_int.obsm['X_pca_harmony'] = Z

    # Build kNN graph and UMAP from Harmony-corrected coordinates
    sc.pp.neighbors(adata_int, use_rep='X_pca_harmony', key_added='neighbors_harmony')
    sc.tl.umap(adata_int, neighbors_key='neighbors_harmony')
    adata_int.obsm['X_umap_harmony'] = adata_int.obsm['X_umap'].copy()
    sc.tl.leiden(
        adata_int, resolution=0.5,
        neighbors_key='neighbors_harmony',
        key_added='leiden_harmony',
    )
    print("Harmony done.")


In [ ]:
if _integration_ok:
    # ── scVI integration ──────────────────────────────────────────────────────
    # scVI trains a variational autoencoder on raw counts, learning a
    # batch-corrected latent space directly from the data.

    print("Setting up scVI model...")
    scvi.settings.seed = RANDOM_SEED

    # scVI requires the raw integer counts stored in a layer
    scvi.model.SCVI.setup_anndata(adata_int, layer='counts', batch_key='batch')
    _model = scvi.model.SCVI(adata_int, n_layers=2, n_latent=30)

    print("Training scVI (this may take ~1-2 min on CPU; much faster on GPU)...")
    _model.train(max_epochs=100, train_size=0.9, early_stopping=True)

    # Extract latent embedding and build UMAP from it
    adata_int.obsm['X_scVI'] = _model.get_latent_representation()
    sc.pp.neighbors(adata_int, use_rep='X_scVI', key_added='neighbors_scvi')
    sc.tl.umap(adata_int, neighbors_key='neighbors_scvi')
    adata_int.obsm['X_umap_scvi'] = adata_int.obsm['X_umap'].copy()
    sc.tl.leiden(
        adata_int, resolution=0.5,
        neighbors_key='neighbors_scvi',
        key_added='leiden_scvi',
    )
    print("scVI done.")


In [ ]:
if _integration_ok:
    # ── Side-by-side UMAP comparison ─────────────────────────────────────────
    # Three rows x two columns:
    #   row 1 - Uncorrected (baseline)
    #   row 2 - Harmony-corrected
    #   row 3 - scVI-corrected
    # Left column: coloured by batch (should MIX after integration)
    # Right column: coloured by Leiden cluster

    _configs = [
        ('X_umap_uncorrected', 'batch', 'leiden_uncorrected', 'Uncorrected'),
        ('X_umap_harmony',     'batch', 'leiden_harmony',     'Harmony'),
        ('X_umap_scvi',        'batch', 'leiden_scvi',        'scVI'),
    ]
    fig, axes = plt.subplots(3, 2, figsize=(14, 16))

    for row, (basis, batch_col, cluster_col, title) in enumerate(_configs):
        sc.pl.embedding(
            adata_int, basis=basis,
            color=batch_col, ax=axes[row, 0], show=False,
            title=f'{title} - by batch',
        )
        sc.pl.embedding(
            adata_int, basis=basis,
            color=cluster_col, ax=axes[row, 1], show=False,
            title=f'{title} - Leiden clusters (res=0.5)',
            legend_loc='on data',
        )

    plt.suptitle(
        'Integration comparison: Uncorrected vs Harmony vs scVI\n'
        '(Left: batch mixing quality | Right: resulting cluster structure)',
        y=1.01, fontsize=13,
    )
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'integration_comparison_umap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n> Good batch correction: left-column UMAPs should show batch_A and batch_B interleaved,")
    print("> not separated into distinct regions.")


## 15. Clustering Resolution Optimization

Choosing the right Leiden resolution is critical:

* **Too low** — clusters are under-resolved; biologically distinct populations are merged
* **Too high** — meaningless over-splitting of homogeneous populations

We sweep a range of resolutions and visualise the results three ways:

1. **Cluster count vs resolution** — when does the number of clusters stabilise?
2. **UMAP grid** — how does the partition change visually across resolutions?
3. **Clustree diagram** — traces how clusters split and merge as resolution increases; stable splits indicate meaningful biological boundaries

The analysis uses the **Harmony-corrected** embedding as the primary integration.


In [ ]:
if _integration_ok:
    # ── Resolution sweep ─────────────────────────────────────────────────────
    _resolutions = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0, 1.2]

    print("Running Leiden at multiple resolutions (Harmony)...")
    for _res in _resolutions:
        sc.tl.leiden(
            adata_int,
            resolution=_res,
            neighbors_key='neighbors_harmony',
            key_added=f'leiden_harmony_r{_res}',
            flavor='igraph',
            n_iterations=2,
            directed=False,
        )
        _n = adata_int.obs[f'leiden_harmony_r{_res}'].nunique()
        print(f"  res={_res:.1f} -> {_n} clusters")


In [ ]:
if _integration_ok:
    # ── Cluster count vs resolution line plot ─────────────────────────────────
    _n_clusters = [
        adata_int.obs[f'leiden_harmony_r{r}'].nunique() for r in _resolutions
    ]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(_resolutions, _n_clusters, 'o-', color='steelblue', linewidth=2, markersize=7)
    for _x, _y in zip(_resolutions, _n_clusters):
        ax.annotate(str(_y), (_x, _y), textcoords='offset points', xytext=(0, 6),
                    ha='center', fontsize=8)
    ax.set_xlabel('Leiden Resolution', fontsize=12)
    ax.set_ylabel('Number of Clusters', fontsize=12)
    ax.set_title('Cluster Count vs Resolution (Harmony integration)', fontsize=13)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'resolution_sweep_cluster_count.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
if _integration_ok:
    # ── UMAP grid: clustering at each resolution ──────────────────────────────
    _sel_res = [0.2, 0.3, 0.4, 0.5, 0.6, 0.8]
    _ncols = 3
    _nrows = int(np.ceil(len(_sel_res) / _ncols))

    fig, axes = plt.subplots(_nrows, _ncols, figsize=(16, _nrows * 5))
    axes = axes.flatten()

    for _i, _res in enumerate(_sel_res):
        _key = f'leiden_harmony_r{_res}'
        _n = adata_int.obs[_key].nunique()
        sc.pl.embedding(
            adata_int, basis='X_umap_harmony',
            color=_key, ax=axes[_i], show=False,
            title=f'res={_res}  ({_n} clusters)',
            legend_loc='on data',
        )

    for _j in range(len(_sel_res), len(axes)):
        axes[_j].set_visible(False)

    plt.suptitle('Harmony: Leiden Clustering at Multiple Resolutions', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'resolution_umap_grid.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
if _integration_ok:
    # ── Clustree diagram ──────────────────────────────────────────────────────
    # Each node = one Leiden cluster at a given resolution (node area proportional
    # to cell count). Each directed edge connects a parent cluster to the child
    # cluster(s) it flows into; edge width proportional to fraction of cells that moved.
    # Stable tree-like splits (each parent -> few children) indicate genuine
    # biological structure; erratic re-merging suggests noise.

    def plot_clustree(adata, resolutions, prefix='leiden_harmony_r',
                      figsize=(14, 8), min_edge_fraction=0.05):
        # Build directed graph
        node_meta = {}   # node_id -> (res_index, cluster_label, cell_count)
        edges     = []   # (src_id, dst_id, fraction)

        for ri, res in enumerate(resolutions):
            key = f'{prefix}{res}'
            for cl in adata.obs[key].astype(str).unique():
                nid   = f'{ri}_{cl}'
                count = int((adata.obs[key] == cl).sum())
                node_meta[nid] = (ri, cl, count)

            if ri < len(resolutions) - 1:
                nxt_key = f'{prefix}{resolutions[ri + 1]}'
                for cl in adata.obs[key].astype(str).unique():
                    mask  = adata.obs[key] == cl
                    total = mask.sum()
                    for ncl, cnt in adata.obs.loc[mask, nxt_key].value_counts().items():
                        frac = cnt / total
                        if frac >= min_edge_fraction:
                            edges.append((f'{ri}_{cl}', f'{ri + 1}_{ncl}', frac))

        # Layout: x = resolution index, y = cluster position within each level
        pos = {}
        for nid, (ri, cl, _) in node_meta.items():
            same   = sorted([n for n, (r, _, _) in node_meta.items() if r == ri],
                             key=lambda n: int(n.split('_')[1]))
            n_same = len(same)
            ci     = same.index(nid)
            pos[nid] = (ri, ci - n_same / 2.0)

        max_count = max(m[2] for m in node_meta.values())
        cmap      = plt.cm.tab20
        fig, ax   = plt.subplots(figsize=figsize)

        # Draw edges first (behind nodes)
        for src, dst, w in edges:
            xs, ys = pos[src]
            xd, yd = pos[dst]
            ax.annotate('', xy=(xd, yd), xytext=(xs, ys),
                         arrowprops=dict(arrowstyle='->', color='#888888',
                                         lw=w * 6, alpha=0.55,
                                         connectionstyle='arc3,rad=0.05'))

        # Draw nodes
        for nid, (ri, cl, count) in node_meta.items():
            x, y  = pos[nid]
            size  = 200 + (count / max_count) * 1200
            color = cmap(int(cl) % 20)
            ax.scatter(x, y, s=size, color=color, zorder=3,
                        edgecolors='white', linewidths=1.5)
            ax.text(x, y, cl, ha='center', va='center', fontsize=7,
                    fontweight='bold', color='white', zorder=4)

        ax.set_xticks(range(len(resolutions)))
        ax.set_xticklabels([str(r) for r in resolutions], fontsize=10)
        ax.set_xlabel('Leiden Resolution', fontsize=12)
        ax.set_yticks([])
        ax.set_title('Clustree: Leiden cluster hierarchy (Harmony integration)', fontsize=13)
        ax.spines[['top', 'right', 'left']].set_visible(False)
        plt.tight_layout()
        return fig

    _fig_ct = plot_clustree(adata_int, _resolutions)
    _fig_ct.savefig(FIGURES_DIR / 'clustree_harmony.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n> Interpret: node area = cell count, edge width = fraction of cells that move.")
    print("> Stable tree-like splits -> genuine biology; erratic merging -> noise or over-resolution.")


## 16. Pathway Enrichment Analysis: ORA & GSEA

Two complementary enrichment strategies help interpret cluster marker genes in terms of known biology:

| Method | Input | Test | Best for |
|--------|-------|------|---------|
| **ORA** (Over-Representation Analysis) | Significant upregulated genes per cluster (pval_adj < 0.05, logFC > 0) | Hypergeometric / Fisher's exact | Focused query when you trust your significance cutoff |
| **GSEA** (Gene Set Enrichment Analysis) | Full ranked gene list (Wilcoxon score) | Kolmogorov-Smirnov statistic | More sensitive; no arbitrary cutoff needed |

We use **`gseapy`** which provides pre-built gene-set libraries directly (no local files needed):

* `MSigDB_Hallmark_2020` — 50 well-defined hallmark gene sets (cancer / immune / metabolism)
* `GO_Biological_Process_2021` — Gene Ontology biological processes
* `KEGG_2021_Human` — KEGG metabolic and signalling pathways

> **Install:** `pip install gseapy`
>
> **Note:** The analysis uses Harmony Leiden clusters at `leiden_harmony_r0.5` by default.
> Adjust `_cluster_key` to match your chosen resolution from §16.


In [ ]:
try:
    import gseapy as gp
    _gseapy_ok = True
except ImportError:
    print("Install gseapy: pip install gseapy")
    _gseapy_ok = False

if _gseapy_ok and _integration_ok:
    # ── Run one-vs-rest Wilcoxon marker analysis on Harmony clusters ──────────
    _cluster_key = 'leiden_harmony_r0.5'  # adjust to your chosen resolution from §16
    _mk_key      = f'rank_genes_{_cluster_key}'

    if _mk_key not in adata_int.uns:
        print(f"Running Wilcoxon one-vs-rest for {_cluster_key}...")
        sc.tl.rank_genes_groups(
            adata_int,
            groupby=_cluster_key,
            method='wilcoxon',
            key_added=_mk_key,
            use_raw=False,
        )
        print("Done.")
    else:
        print(f"Using existing marker results: {_mk_key}")

    # ── Gene-set libraries to query (no download required; gseapy fetches them) ─
    _gene_set_libraries = {
        'MSigDB_Hallmark': 'MSigDB_Hallmark_2020',
        'GO_BP':           'GO_Biological_Process_2021',
        'KEGG':            'KEGG_2021_Human',
    }
    print("Gene-set libraries:", list(_gene_set_libraries.keys()))


In [ ]:
if _gseapy_ok and _integration_ok:
    # ── ORA: test significant upregulated markers for each cluster ────────────
    import os as _os

    _ora_padj_cutoff  = 0.05
    _ora_logfc_cutoff = 0.0   # keep genes with logFC > 0 (upregulated)

    _mk      = adata_int.uns[_mk_key]
    _groups  = _mk['names'].dtype.names
    _n_genes = _mk['names'].shape[0]

    # Background = all genes in the marker analysis
    _background = sorted({
        _mk['names'][g][r] for g in _groups for r in range(_n_genes)
    })
    print(f"ORA background: {len(_background)} genes")

    _ora_out_dir = here('results', 'enrichment', 'ORA')
    _ora_out_dir.mkdir(parents=True, exist_ok=True)

    _all_ora = {lib: [] for lib in _gene_set_libraries}

    for _cl in _groups:
        _sig_genes = [
            _mk['names'][_cl][r]
            for r in range(_n_genes)
            if (_mk['pvals_adj'][_cl][r] < _ora_padj_cutoff and
                _mk['logfoldchanges'][_cl][r] > _ora_logfc_cutoff)
        ]
        print(f"  Cluster {_cl}: {len(_sig_genes)} significant upregulated genes")
        if not _sig_genes:
            continue

        for _lib_name, _lib_id in _gene_set_libraries.items():
            try:
                _enr = gp.enrich(
                    gene_list=_sig_genes,
                    gene_sets=_lib_id,
                    background=_background,
                    outdir=None,
                    verbose=False,
                )
                _df = _enr.results.copy()
                _df.insert(0, 'cluster', _cl)
                _df['query_size'] = len(_sig_genes)
                if 'Overlap' in _df.columns:
                    _ov = _df['Overlap'].str.split('/')
                    _df['count']      = _ov.str[0].astype(int)
                    _df['gene_ratio'] = _df['count'] / len(_sig_genes)
                _all_ora[_lib_name].append(_df)
            except Exception as _exc:
                print(f"    ORA {_lib_name} cluster {_cl}: {_exc}")

    # ── Save combined ORA CSVs ────────────────────────────────────────────────
    for _lib_name, _dfs in _all_ora.items():
        if not _dfs:
            continue
        _combined = pd.concat(_dfs, ignore_index=True)
        _out = _ora_out_dir / f'ORA_{_lib_name}_{_cluster_key}.csv'
        _combined.to_csv(_out, index=False)
        _n_sig = (_combined['Adjusted P-value'] < _ora_padj_cutoff).sum() if 'Adjusted P-value' in _combined.columns else 0
        print(f"  {_lib_name}: {len(_combined)} rows | {_n_sig} significant -> {_out.name}")


In [ ]:
if _gseapy_ok and _integration_ok:
    # ── ORA dotplot: top-8 terms per cluster ─────────────────────────────────
    import re as _re

    def _fmt_label(t, max_len=45):
        t = _re.sub(r'\s*\([^)]*\)\s*$', '', str(t)).strip()
        return t[:max_len] if len(t) > max_len else t

    _adj_col  = 'Adjusted P-value'
    _term_col = 'Term'
    _n_top    = 8

    for _lib_name, _dfs in _all_ora.items():
        if not _dfs:
            continue
        _combined = pd.concat(_dfs, ignore_index=True)
        if _combined.empty or _adj_col not in _combined.columns:
            continue

        _clust_ids = sorted(_combined['cluster'].unique(), key=int)
        _ncols = min(3, len(_clust_ids))
        _nrows = int(np.ceil(len(_clust_ids) / _ncols))
        _fig, _axes = plt.subplots(
            _nrows, _ncols,
            figsize=(_ncols * 6, _nrows * max(3, _n_top * 0.45 + 1.5)),
            constrained_layout=True,
        )
        _axes = np.array(_axes).flatten()

        for _i, _cid in enumerate(_clust_ids):
            _cdf = (_combined[_combined['cluster'] == _cid]
                    .sort_values(_adj_col)
                    .head(_n_top)
                    .copy()
                    .sort_values('gene_ratio' if 'gene_ratio' in _combined.columns else _adj_col,
                                 ascending=True))
            _ax = _axes[_i]
            if _cdf.empty:
                _ax.set_visible(False)
                continue
            _cnt   = _cdf['count'].values.astype(float) if 'count' in _cdf.columns else np.full(len(_cdf), 30.0)
            _rng   = _cnt.max() - _cnt.min() if _cnt.max() > _cnt.min() else 1.0
            _sizes = 30 + (_cnt - _cnt.min()) / _rng * 270
            _gr    = _cdf['gene_ratio'].values if 'gene_ratio' in _cdf.columns else np.arange(len(_cdf))
            _sc    = _ax.scatter(_gr, np.arange(len(_cdf)),
                                 c=_cdf[_adj_col].values, cmap='RdYlBu_r',
                                 vmin=0, vmax=_ora_padj_cutoff, s=_sizes, zorder=3)
            _ax.set_yticks(np.arange(len(_cdf)))
            _ax.set_yticklabels([_fmt_label(t) for t in _cdf[_term_col]], fontsize=7)
            _ax.set_xlabel('GeneRatio', fontsize=9)
            _ax.set_title(f'Cluster {_cid}', fontsize=10)
            _ax.grid(axis='x', linestyle='--', alpha=0.4)
            _fig.colorbar(_sc, ax=_ax, fraction=0.04, pad=0.01).set_label('p.adjust', fontsize=7)

        for _j in range(len(_clust_ids), len(_axes)):
            _axes[_j].set_visible(False)

        _fig.suptitle(f'ORA - {_lib_name}  ({_cluster_key})', fontsize=12)
        _out_fig = here('results', 'enrichment', 'ORA') / f'ORA_{_lib_name}_dotplot.png'
        _fig.savefig(_out_fig, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"  Saved: {_out_fig.name}")


In [ ]:
if _gseapy_ok and _integration_ok:
    # ── GSEA (pre-ranked): rank all genes by Wilcoxon score per cluster ───────
    # Unlike ORA, no significance cutoff is applied before ranking.
    # The full ranked gene list provides greater sensitivity, especially
    # when many genes have borderline p-values.

    _gsea_out_dir = here('results', 'enrichment', 'GSEA')
    _gsea_out_dir.mkdir(parents=True, exist_ok=True)

    _all_gsea = {lib: [] for lib in _gene_set_libraries}

    for _cl in _groups:
        # Build pre-ranked series: gene -> Wilcoxon score (descending)
        _genes  = [_mk['names'][_cl][r]  for r in range(_n_genes)]
        _scores = [float(_mk['scores'][_cl][r]) for r in range(_n_genes)]
        _rnk = (pd.Series(_scores, index=_genes)
                  .replace([np.inf, -np.inf], np.nan)
                  .dropna()
                  .sort_values(ascending=False))
        _rnk = _rnk[~_rnk.index.duplicated(keep='first')]
        print(f"  Cluster {_cl}: {len(_rnk)} genes ranked")

        for _lib_name, _lib_id in _gene_set_libraries.items():
            try:
                _pre = gp.prerank(
                    rnk=_rnk,
                    gene_sets=_lib_id,
                    outdir=None,
                    permutation_num=100,
                    seed=RANDOM_SEED,
                    threads=4,
                    verbose=False,
                )
                _df = _pre.res2d.copy()
                _df.insert(0, 'cluster', _cl)
                _all_gsea[_lib_name].append(_df)
            except Exception as _exc:
                print(f"    GSEA {_lib_name} cluster {_cl}: {_exc}")

    # ── Save combined GSEA CSVs ───────────────────────────────────────────────
    for _lib_name, _dfs in _all_gsea.items():
        if not _dfs:
            continue
        _combined = pd.concat(_dfs, ignore_index=True)
        _out = _gsea_out_dir / f'GSEA_{_lib_name}_{_cluster_key}.csv'
        _combined.to_csv(_out, index=False)
        _fdr_col = 'FDR q-val'
        _n_sig = (_combined[_fdr_col] < 0.25).sum() if _fdr_col in _combined.columns else 0
        print(f"  {_lib_name}: {len(_combined)} rows | {_n_sig} with FDR<0.25 -> {_out.name}")


In [ ]:
if _gseapy_ok and _integration_ok:
    # ── GSEA dotplot: top-8 terms by NES per cluster ──────────────────────────
    # x = NES (Normalised Enrichment Score)
    #   > 0: gene set is enriched among genes upregulated in this cluster
    #   < 0: gene set is enriched among genes downregulated in this cluster
    # dot size = number of leading-edge genes
    # dot colour = FDR q-value (lower = more significant)

    _nes_col    = 'NES'
    _fdr_col    = 'FDR q-val'
    _fdr_cutoff = 0.25
    _n_top_gsea = 8

    for _lib_name, _dfs in _all_gsea.items():
        if not _dfs:
            continue
        _combined = pd.concat(_dfs, ignore_index=True)
        _plot_df  = (_combined[_combined[_fdr_col] < _fdr_cutoff].copy()
                     if _fdr_col in _combined.columns else _combined.copy())
        if _plot_df.empty or _nes_col not in _plot_df.columns:
            print(f"{_lib_name}: no significant GSEA terms (FDR < {_fdr_cutoff}) to plot.")
            continue

        if 'Lead_genes' in _plot_df.columns:
            _plot_df['lead_count'] = _plot_df['Lead_genes'].fillna('').apply(
                lambda x: len(str(x).split(';')) if x else 0
            )
        else:
            _plot_df['lead_count'] = 50

        _clust_ids = sorted(_plot_df['cluster'].unique(), key=int)
        _ncols = min(3, len(_clust_ids))
        _nrows = int(np.ceil(len(_clust_ids) / _ncols))
        _fig, _axes = plt.subplots(
            _nrows, _ncols,
            figsize=(_ncols * 6, _nrows * max(3, _n_top_gsea * 0.45 + 1.5)),
            constrained_layout=True,
        )
        _axes = np.array(_axes).flatten()

        for _i, _cid in enumerate(_clust_ids):
            _cdf = (_plot_df[_plot_df['cluster'] == _cid]
                    .sort_values(_nes_col, ascending=False)
                    .head(_n_top_gsea)
                    .copy()
                    .sort_values(_nes_col, ascending=True))
            _ax = _axes[_i]
            if _cdf.empty:
                _ax.set_visible(False)
                continue
            _lc    = _cdf['lead_count'].values.astype(float)
            _rng   = _lc.max() - _lc.min() if _lc.max() > _lc.min() else 1.0
            _sizes = 30 + (_lc - _lc.min()) / _rng * 270
            _vmin  = _cdf[_fdr_col].min()
            _vmax  = max(_cdf[_fdr_col].max(), _fdr_cutoff)
            _sc = _ax.scatter(
                _cdf[_nes_col].values, np.arange(len(_cdf)),
                c=_cdf[_fdr_col].values, cmap='RdYlBu_r',
                vmin=_vmin, vmax=_vmax, s=_sizes, zorder=3,
            )
            _ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
            _ax.set_yticks(np.arange(len(_cdf)))
            _ax.set_yticklabels([_fmt_label(t) for t in _cdf['Term']], fontsize=7)
            _ax.set_xlabel('NES', fontsize=9)
            _ax.set_title(f'Cluster {_cid}', fontsize=10)
            _ax.grid(axis='x', linestyle='--', alpha=0.4)
            _fig.colorbar(_sc, ax=_ax, fraction=0.04, pad=0.01).set_label('FDR q-val', fontsize=7)

        for _j in range(len(_clust_ids), len(_axes)):
            _axes[_j].set_visible(False)

        _fig.suptitle(
            f'GSEA - {_lib_name}  (FDR<{_fdr_cutoff})  ({_cluster_key})', fontsize=12
        )
        _out_fig = here('results', 'enrichment', 'GSEA') / f'GSEA_{_lib_name}_dotplot.png'
        _fig.savefig(_out_fig, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"  Saved: {_out_fig.name}")
        print("  > Positive NES: gene set enriched among upregulated cluster markers")
        print("  > Negative NES: gene set enriched among downregulated markers")


## 17. Save Results


In [ ]:
# Save the annotated AnnData object to the project data/ directory
adata.write(results_file)
print('Saved to', results_file)


## Workshop Exercises

### Beginner
1. **Change QC thresholds** — try `pct_counts_mt < 10` instead of 5. How does it affect cell numbers and clusters?
2. **Change Leiden resolution** — try `resolution=2.0`. Does the additional granularity make biological sense?
3. **Visualise a new marker gene** — look up another PBMC marker and plot it on the UMAP.

### Intermediate
4. **Ribosomal filter** — filter cells with `pct_counts_ribo > 50`. Does this change anything?
5. **Compare UMAP and t-SNE** — what structural differences do you observe?
6. **Regress out cell cycle** — add `S_score` and `G2M_score` to the `regress_out` call. Does this flatten the cycling population?

### Advanced
7. **Alternative HVG method** — use `flavor='seurat_v3'` in `sc.pp.highly_variable_genes`. Compare the selected genes.
8. **Adjust batch effect strength (§15)** — change the Poisson lambda in the artificial batch step and observe how clustering changes.
9. **Choose your resolution (§15/16)** — use the clustree from §16 to pick a biologically meaningful resolution. Do Harmony and scVI agree?
10. **BBKNN batch correction** — install `bbknn` and run `sc.external.pp.bbknn`. Compare with Harmony and scVI UMAPs.
11. **Enrich your own cluster (§17)** — pick a cluster. What are its top ORA GO:BP terms? Do the GSEA NES values agree?
12. **Pseudotime** — use `sc.tl.diffmap`/`sc.tl.dpt` to compute pseudotime ordering of T cells.

### Reflection questions
- Why do we store `adata.raw` before subsetting to HVGs?
- What happens if you skip the `regress_out` step?
- Why does the number of PCs used for the neighbour graph matter?
- When would you prefer `sc.tl.ingest` over joint batch correction methods like Harmony or scVI?
- What is the difference between ORA and GSEA? When does GSEA have an advantage?
- How do you interpret a positive vs negative NES in the GSEA results?
